# 🔴 Interacting with Redis using Python

This notebook is a didactic and practical guide that demonstrates how to interact with **Redis** (a **Key-Value** NoSQL database) using the Python language.

## 🛠️ What is Redis?
Redis (Remote Dictionary Server) is an in-memory data structure store, used as a database, cache, and message broker. It is extremely fast because it keeps data in **RAM**, persisting it to disk asynchronously.

### Conceptual Summary

| Property | Details |
|---|---|
| **Paradigm** | Key-Value Store |
| **Query Language** | Redis Commands (SET, GET, DEL, HSET, LPUSH, etc.) |
| **Storage** | Data in RAM with optional disk persistence (RDB/AOF) |
| **When to use** | High-performance cache, user sessions, message queues, real-time counters, leaderboards, pub/sub |
| **When NOT to use** | Complex relational data, queries with JOINs, data exceeding available RAM, persistence as a critical requirement |

### Local Connection Details (Docker Compose):
- **Host:** `localhost`
- **Port:** `6379`
- **Authentication:** None (default local configuration)

## 📋 Prerequisites

Before running this notebook, make sure that:

1. **Docker** is installed and running on your machine.
2. The project containers have been started with `make up` or `docker compose up -d`.
3. The `redis` container is running (check with `docker compose ps`).

> **💡 Tip:** Redis starts almost instantly, unlike other databases like Cassandra which can take 1-2 minutes.

## 2. Connecting to the Database
Let's import the library and create a connection instance. 

> **💡 Key Concept:** The `decode_responses=True` parameter is essential! By default, Redis returns data as **bytes** (`b'text'`). When this parameter is enabled, responses are automatically converted to regular Python **strings**, making manipulation easier.

**Expected output:**
```
✅ Connection to Redis established successfully!
```

In [ ]:
import redis

# Create connection with the local Redis client
# - host: server address (localhost because the Docker container maps the port)
# - port: default Redis port
# - decode_responses: converts bytes to strings automatically
try:
    client = redis.Redis(host='localhost', port=6379, decode_responses=True)
    
    # The ping() command tests if the connection is working
    # Returns True if the server responded with PONG
    if client.ping():
        print("✅ Connection to Redis established successfully!")
except Exception as e:
    print(f"❌ Error connecting to Redis: {e}")
    print("Make sure the Redis container is running (use 'make up' or 'docker compose up -d')")

---
## 3. Basic CRUD Operations (String Keys)
The most basic value type in Redis is **String** (which can contain text, numbers, or even serialized binary data).

> **💡 Key Concept:** In Redis, the key naming convention uses `:` (colons) as a logical separator to create a visual hierarchy. Example: `usuario:1:nome` indicates it is the `nome` field of `usuario` with ID `1`. This is just a convention — Redis treats it as a single string.

### 3.1 CREATE — Inserting data
The `SET` command stores a value associated with a key.

Syntax: 

```python
client.set(<key>,<value>)
```

**Expected output:**
```
✍️ Key 'usuario:1:nome' created with value: 'Carlos Silva'
```

In [ ]:
# SET defines the value of a key
# If the key does not exist, it will be created automatically
client.set('usuario:1:nome', 'Carlos Silva')
print("Data inserted")

### 3.2 READ — Reading data
The `GET` command retrieves the value stored in a key. Returns `None` if the key does not exist.

Syntax:

```python
client.get(<key>) # -> returns the value
```

**Expected output:**
```
📖 Value of 'usuario:1:nome': Carlos Silva
📖 Value of 'chave:inexistente': None
```

In [ ]:
# GET retrieves the value associated with the key
nome_usuario = client.get('usuario:1:nome')
print(f"📖 Value of 'usuario:1:nome': {nome_usuario}")

# If we try to read a key that does not exist, the return will be None
valor_inexistente = client.get('chave:inexistente')
print(f"📖 Value of 'chave:inexistente': {valor_inexistente}")

### 3.3 UPDATE — Updating data
In Redis, **there is no separate UPDATE command**. A new `SET` on the same key simply overwrites the old value. This simplicity is one of the characteristics of the key-value model.

**Expected output:**
```
🔄 Value BEFORE update: Carlos Silva
🔄 Value AFTER update: Carlos Souza
```

In [ ]:
# Remember: there is no "UPDATE" in Redis — SET overwrites
print(f"🔄 Value BEFORE update: {client.get('usuario:1:nome')}")

client.set('usuario:1:nome', 'Carlos Souza')

print(f"🔄 Value AFTER update: {client.get('usuario:1:nome')}")

### 3.4 DELETE — Deleting data
The `DEL` command removes a key and its value from the database. The `EXISTS` command can be used to check if a key exists (returns `1` if it exists, `0` otherwise).

Syntax:

```python
client.delete(<key>)
client.exists(<key>)
```

**Expected output:**
```
🗑️ Key deleted successfully.
❓ Does the key 'usuario:1:nome' still exist? No
```

In [ ]:
# DEL removes the key from the database
client.delete('usuario:1:nome')
print("🗑️ Key deleted successfully.")

# EXISTS checks if the key is still present
# Returns 1 (True) if it exists, 0 (False) if it does not exist
existe = client.exists('usuario:1:nome')
print(f"❓ Does the key 'usuario:1:nome' still exist? {'Yes' if existe else 'No'}")

---
## 4. Expiration Control (TTL — Time to Live)
One of the most powerful features of Redis is the ability to set an **automatic expiration** for keys. After the time expires, the key is automatically removed from the database.

> **💡 Real Use Case:** TTL is widely used to manage **user sessions** (e.g., JWT tokens that expire in 30 minutes), **temporary cache** (e.g., result of an API that changes every 5 minutes) and **rate limiting** (e.g., limit 100 requests per minute per IP).

**Syntax: SET with expiration -> SETEX**

```phthon
client.setex(<key>, <time_seconds>, <value>) # Insert key-value with expiration time
```

**Syntax: TTL returns remaining time in seconds**


```phthon
client.ttl(<key>) # Insert key-value with expiration time
```

In [ ]:
import time

# SETEX = SET with EXpiration
# Parameters: (chave, tempo_em_segundos, valor)
# The key will be automatically removed after the defined time
client.setex('sessao:token', 5, 'jwt_token_exemplo_123')
print("🔑 Session token inserted with a 5-second expiration.")

# TTL (Time To Live) returns the remaining life time of the key in seconds
# Returns -1 if the key has no expiration, -2 if the key does not exist
ttl_inicial = client.ttl('sessao:token')
print(f"⏳ Initial TTL: {ttl_inicial} seconds")

# Wait 30 seconds to see the TTL decrease
time.sleep(30)
ttl_restante = client.ttl('sessao:token')
valor_token = client.get('sessao:token')
print(f"⏳ TTL after 30 seconds: {ttl_restante} seconds (Value: {valor_token})")

# Wait another 60 seconds (totaling 6, exceeding the 60 seconds of TTL)
print("Waiting for the key to expire...")
time.sleep(60)
valor_expirado = client.get('sessao:token')
print(f"🗑️ Value retrieved after expiration: {valor_expirado} (Key expired automatically!)")

---
## 5. Advanced Data Structures
Redis goes far beyond simple strings! It supports several native data structures, which makes it extremely versatile. Let's explore the three most commonly used:

| Structure | Description | Typical Use Case |
|---|---|---|
| **Hashes** | Dictionaries (field-value pairs within a key) | User profiles, structured objects |
| **Lists** | Lists ordered by insertion order | Task queues, action history |
| **Sets** | Collections of unique values (no duplicates) | Tags, followers, presence list |

### 5A. Hashes (Dictionaries/Objects)
Hashes are great for representing **structured objects**, containing multiple fields and values within a single main key. Think of them as a "mini Python dictionary" stored in Redis.

> **💡 Advantage over Strings:** Instead of creating multiple keys (`usuario:100:nome`, `usuario:100:email`, `usuario:100:idade`), you can store everything in a single Hash under the key `usuario:100`. This is more efficient in memory and organization.

**Sintaxa HSET - hash set**

```python
client.hset(<main_key>, mapping=<dictionary>)
```

In [ ]:
chave_hash = 'usuario:100'

# HSET with mapping inserts multiple fields at once into the Hash
# It is equivalent to running multiple individual HSETs
client.hset(chave_hash, mapping={
    'nome': 'Alice Silva',
    'email': 'alice@email.com',
    'idade': '28',
    'cidade': 'João Pessoa'
})
print(f"📝 Hash created in '{chave_hash}'")

**Syntax: HGET -> hash get**

```python
client.hget(<main_key>, <dict_key>)
```

In [ ]:
# HGET gets the value of a specific field from the Hash
nome = client.hget(chave_hash, 'nome')
print(f"👤 User name: {nome}")





**Syntax: HGETALL -> hash get all**

```python
client.hgetall(<main_key>)
```

In [ ]:
# HGETALL returns all fields and values from the Hash as a Python dictionary
dados_usuario = client.hgetall(chave_hash)
print(f"📦 Complete object: {dados_usuario}")

**Syntax: HSET -> hash SET**

```python
client.hset(<hash_key>, <dict_key>, <new_dict_value>)
```

In [ ]:
# HSET on an existing field updates only that field (without affecting the others)
client.hset(chave_hash, 'idade', '29')
idade_atualizada = client.hget(chave_hash, 'idade')
print(f"🔄 Age updated to: {idade_atualizada}")

**Syntax: HDEL -> hash DEL**

```python
client.hdel(<hash_key>, <dict_key>) # removes key-value from dict
```

In [ ]:
# HDEL removes a specific field from the Hash (without deleting the other fields)
client.hdel(chave_hash, 'cidade')
dados_finais = client.hgetall(chave_hash)
print(f"🗑️ Hash after deleting the field 'cidade': {dados_finais}")



**Syntax: delete**

```python
client.delete(<hash_key>) # removes the entire object
```

In [ ]:
# Clear key to keep environment organized
client.delete(chave_hash)

### 5B. Lists (Queue / Stack)
Redis Lists are collections of strings **ordered by insertion order**. 

You can add elements at the beginning (`LPUSH`) or at the end (`RPUSH`), making them useful as **queues (FIFO)** or **stacks (LIFO)**.

> **💡 Analogy:** Imagine a bank queue. New people join at the end of the line (`RPUSH`), and the next person to be served comes from the front (`LPOP`). If someone has priority, they can "cut in line" and enter at the front (`LPUSH`).

**Expected output:**
```
📋 Task list size: 3
🔍 Complete list (note the order): ['Corrigir bug crítico em produção', 'Enviar e-mail para o cliente', 'Revisar PR pendente']
✅ Task completed and removed from queue: 'Corrigir bug crítico em produção'
🔍 Remaining list: ['Enviar e-mail para o cliente', 'Revisar PR pendente']
```

In [ ]:
chave_lista = 'tarefas:urgentes'
client.delete(chave_lista)  # Clear if it already exists from previous executions

# RPUSH adds items at the END of the list (Right Push)
client.rpush(chave_lista, 'Enviar e-mail para o cliente')
client.rpush(chave_lista, 'Revisar PR pendente')

# LPUSH adds items at the START of the list (Left Push)
# This simulates an urgent task "cutting in line"
client.lpush(chave_lista, 'Corrigir bug crítico em produção')

# LLEN returns the size (length) of the list
tamanho = client.llen(chave_lista)
print(f"📋 Task list size: {tamanho}")

# LRANGE returns a subset of the list (0 = first, -1 = last)
# Using 0 to -1 we get the full list
itens = client.lrange(chave_lista, 0, -1)
print(f"🔍 Complete list (note the order): {itens}")


In [ ]:

# LPOP removes and returns the FIRST item from the list
# This simulates "serving the next in line"
primeira_tarefa = client.lpop(chave_lista)
print(f"✅ Task completed and removed from queue: '{primeira_tarefa}'")

# Check the remaining state of the list
itens_restantes = client.lrange(chave_lista, 0, -1)
print(f"🔍 Remaining list: {itens_restantes}")

# Clear key
client.delete(chave_lista)

### 5C. Sets (Unique and Unordered Collections)
Sets are collections of **unique** (no duplicates) and **unordered** strings. Useful for tags, attendance lists, followers, etc.

> **💡 Key Concept:** The main difference between Sets and Lists is that Sets **ignore duplicates automatically**. If you try to add a value that already exists, it simply won't be duplicated.

**Expected output:**
```
🏷️ Post Tags (notice that 'nosql' appears only once): {'programacao', 'nosql', 'tecnologia'}
❓ Contains the tag 'python'? No
❓ Contains the tag 'nosql'? Yes
🗑️ Remaining tags after removing 'tecnologia': {'programacao', 'nosql'}
```

In [ ]:
chave_set = 'tags:post:42'
client.delete(chave_set)  # Clear if it already exists from previous executions

# SADD adds members to the Set
client.sadd(chave_set, 'tecnologia')
client.sadd(chave_set, 'programacao')
client.sadd(chave_set, 'nosql')

# Try to add a duplicate item — it will have no effect!
client.sadd(chave_set, 'nosql')

# SMEMBERS returns all members of the Set
# Note: the order may vary since Sets are unordered
membros = client.smembers(chave_set)
print(f"🏷️ Post Tags (notice that 'nosql' appears only once): {membros}")

# SISMEMBER checks if a value belongs to the Set (returns True/False)
tem_python = client.sismember(chave_set, 'python')
tem_nosql = client.sismember(chave_set, 'nosql')
print(f"❓ Contains the tag 'python'? {'Yes' if tem_python else 'No'}")
print(f"❓ Contains the tag 'nosql'? {'Yes' if tem_nosql else 'No'}")



In [ ]:
# SREM removes a specific member from the Set
client.srem(chave_set, 'tecnologia')
membros_finais = client.smembers(chave_set)
print(f"🗑️ Remaining tags after removing 'tecnologia': {membros_finais}")

# Clear key
client.delete(chave_set)

---
## 6. Closing the Connection
It is good practice to close the connection to Redis at the end of use to free up resources.

In [ ]:
# Close the connection to Redis
client.close()
print("🔌 Connection to Redis closed successfully.")

---
## 🏁 Conclusion
Congratulations! You have completed the introduction to Redis. In this notebook, you learned how to:
- ✅ Connect to Redis in Python using the `redis` library.
- ✅ Save and retrieve strings with expiration control (TTL).
- ✅ Manipulate structured data in **Dictionaries** (Hashes), **Queues/Stacks** (Lists) and **Duplicate-free Sets** (Sets).

### 🚀 Next Steps
To continue deepening your knowledge of Redis, try:
1. **Sorted Sets (ZADD):** Sets ordered by score — ideal for leaderboards and rankings.
2. **Pipelines (`client.pipeline()`):** Group multiple commands into a single request to improve performance.
3. **Pub/Sub (`client.pubsub()`):** Implement a real-time message publishing and subscription system.
4. **Transactions (`client.pipeline(transaction=True)`):** Ensure atomicity in operations involving multiple keys.

### 📚 Useful References
- [Official Redis documentation](https://redis.io/docs/)
- [Command reference](https://redis.io/commands)
- [redis-py (Python library)](https://redis-py.readthedocs.io/)